[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/burzynski-lab/OU_PHYS2222_ComputationalPhysics/blob/main/LectureNotes/05_Matplotlib.ipynb)

# Visualization with Matplotlib

**PHYS 2222: Computational Physics** · University of Oklahoma · Spring 2027 · Week 4

> **Before you start:** in Colab, choose **File → Save a copy in Drive** so your changes are kept.

## Learning goals

By the end of this notebook you should be able to:

- make figures with Matplotlib's object-oriented interface, `fig, ax = plt.subplots()`;
- draw line plots, scatter plots, error bars, and histograms, and control colors, markers, and line styles;
- label axes with units and LaTeX, add legends and titles, set axis limits, and use logarithmic axes;
- arrange several panels in one figure with `plt.subplots(nrows, ncols)`;
- show 2D data (a function of $x$ and $y$) with `imshow`, `pcolormesh`, and `contour`, with a colorbar;
- load real data sets with `np.loadtxt` and `np.genfromtxt` (including missing values), and get a first look at pandas;
- save figures to files, and judge what makes a figure clear and honest.

Physics along the way: classical vs relativistic kinetic energy, blackbody spectra, the photoelectric effect (a first estimate of Planck's constant), the hydrogen spectrum, the sunspot cycle, global temperatures, and a quantum particle in a 2D box.

## 1. Two ways to use Matplotlib

Matplotlib is the standard plotting library in scientific Python. We import its `pyplot` module as `plt` (another universal convention):

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import constants as const

In notebook 04 you used the quick **pyplot style**: call `plt.plot`, `plt.xlabel`, ... and Matplotlib draws on "the current figure" behind the scenes.

In [ ]:
x = np.linspace(0, 2 * np.pi, 200)
plt.plot(x, np.sin(x))
plt.xlabel("x")
plt.ylabel("sin x")
plt.show()

In this course we will use the **object-oriented (OO) style** instead. You ask for a figure and a set of axes explicitly, give them names, and then call methods on them:

- `fig` is the whole figure (the "page");
- `ax` is one set of axes (one panel, with its own $x$ and $y$ axes) inside the figure.

In [ ]:
fig, ax = plt.subplots()
ax.plot(x, np.sin(x))
ax.set_xlabel("x")
ax.set_ylabel("sin x")
plt.show()

It looks the same, so why bother? Because as soon as you have more than one panel, you need to say *which* panel to draw on, and the OO style makes that explicit (`ax1.plot(...)`, `ax2.plot(...)`). It is also what most examples in the Matplotlib documentation and in research code use. The method names are almost the same as the pyplot functions, with `set_` in front of the labelling ones:

| pyplot | object-oriented |
|---|---|
| `plt.plot(...)` | `ax.plot(...)` |
| `plt.xlabel(...)` | `ax.set_xlabel(...)` |
| `plt.title(...)` | `ax.set_title(...)` |
| `plt.xlim(...)` | `ax.set_xlim(...)` |
| `plt.legend()` | `ax.legend()` |

`plt.show()` displays the figure. In Colab the figure appears even without it, but including it is a good habit (in a plain Python script nothing appears without it).

## 2. A complete line plot

**Physics recap.** The kinetic energy of a particle of mass $m$ moving at speed $v = \beta c$ is, according to special relativity,

$$K_\text{rel} = (\gamma - 1)\,mc^2, \qquad \gamma = \frac{1}{\sqrt{1-\beta^2}},$$

while Newtonian mechanics gives $K_\text{cl} = \tfrac12 m v^2 = \tfrac12 mc^2\beta^2$. They agree at low speed, but $K_\text{rel}$ grows without bound as $\beta \to 1$: no finite energy can push a massive particle to the speed of light. A plot shows this far better than a table.

In [ ]:
beta = np.linspace(0, 0.95, 300)
gamma = 1 / np.sqrt(1 - beta**2)
mc2 = const.m_e * const.c**2 / (1e6 * const.e)    # electron rest energy in MeV
K_rel = (gamma - 1) * mc2
K_cl = 0.5 * mc2 * beta**2

fig, ax = plt.subplots()
ax.plot(beta, K_rel, label="relativistic")
ax.plot(beta, K_cl, label="classical")
ax.set_xlabel(r"$\beta = v/c$")
ax.set_ylabel("kinetic energy (MeV)")
ax.set_title("Electron kinetic energy")
ax.legend()
plt.show()

New pieces:

- Calling `ax.plot` twice draws two curves on the same axes; Matplotlib picks a new color automatically.
- `label="..."` names a curve, and `ax.legend()` displays the names.
- **LaTeX in labels:** text between `$...$` is rendered as math, exactly as in Markdown cells. Put an `r` in front of the string (`r"$\beta$"`), a *raw string*, so Python does not treat the backslash as a special character.
- **Units go in the axis label**, usually in parentheses. A label without units is the most common mistake in student figures (and in the rubric for your projects).

### 2.1 Axis limits, grids, and reference lines

`ax.set_xlim(left, right)` and `ax.set_ylim(bottom, top)` zoom in. `ax.axhline(y)` and `ax.axvline(x)` draw horizontal or vertical reference lines across the whole plot. `ax.grid()` adds a light grid.

In [ ]:
fig, ax = plt.subplots()
ax.plot(beta, K_rel / K_cl, label=r"$K_\mathrm{rel} / K_\mathrm{cl}$")
ax.axhline(1.01, color="gray", linestyle="--", label="1% difference")
ax.axhline(1.10, color="gray", linestyle=":", label="10% difference")
ax.set_xlim(0, 0.6)
ax.set_ylim(0.99, 1.4)
ax.set_xlabel(r"$\beta$")
ax.set_ylabel("ratio")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

(The first point, $\beta = 0$, gives $0/0$, so NumPy prints a warning and the ratio there is `nan`, "not a number". Matplotlib simply skips `nan` points. Notebook 06 covers `nan`.)

You can read off roughly where classical mechanics is off by 1% and by 10%. Finding those points precisely is part of Project 1.

### 2.2 Colors, line styles, and markers

`ax.plot` takes many optional keyword arguments. The most useful:

| keyword | examples |
|---|---|
| `color` | `"C0"`...`"C9"` (the default color cycle), `"black"`, `"tab:red"`, `"#1f77b4"` |
| `linestyle` (`ls`) | `"-"`, `"--"`, `":"`, `"-."`, `"none"` |
| `linewidth` (`lw`) | `1`, `2.5` |
| `marker` | `"o"`, `"s"` (square), `"^"`, `"x"`, `"."` |
| `markersize` (`ms`) | `4`, `8` |
| `alpha` | transparency from 0 to 1 |

In [ ]:
t = np.linspace(0, 10, 25)
fig, ax = plt.subplots()
ax.plot(t, np.exp(-t / 3), color="C0", linestyle="-", label="solid line")
ax.plot(t, np.exp(-t / 5), color="C1", linestyle="--", lw=2, label="dashed, thicker")
ax.plot(t, np.exp(-t / 8), color="C2", linestyle="none", marker="o", label="markers only")
ax.plot(t, np.exp(-t / 12), color="C3", marker="s", ms=4, ls=":", label="markers + dotted")
ax.set_xlabel("time (s)")
ax.set_ylabel("fraction remaining")
ax.legend()
plt.show()

**Color choice.** About 1 in 12 men (and 1 in 200 women) have some form of color-vision deficiency, most often red-green. Matplotlib's default colors (`"C0"`, `"C1"`, ...) were chosen to be reasonably distinguishable. Good habits:

- avoid relying on red vs green alone;
- vary line style or marker as well as color, so the figure also works in black and white;
- for continuous color scales use a perceptually uniform colormap such as `"viridis"` (the default), not `"jet"` or `"rainbow"`.

### Your turn 1

Make a plot of the Lorentz factor $\gamma$ versus $\beta$ for $0 \le \beta \le 0.99$. Add a dashed horizontal line at $\gamma = 2$ and a dotted vertical line at the $\beta$ where $\gamma = 2$ (work it out by hand: $\beta = \sqrt{3}/2$). Label both axes and add a legend.

In [ ]:
# Your code here

## 3. Logarithmic axes: blackbody spectra

**Physics recap.** A hot, dense object (a "blackbody") glows with a spectrum that depends only on its temperature $T$. Planck's law gives the spectral radiance per unit wavelength:

$$
B_\lambda(\lambda, T) = \frac{2hc^2}{\lambda^5}\,\frac{1}{e^{hc/(\lambda k_B T)} - 1}.
$$

Hotter objects are brighter at every wavelength, and the peak moves to shorter wavelength (Wien's law, $\lambda_\text{max} T \approx 2.9\times10^{-3}$ m K). Let's plot several temperatures.

In [ ]:
def planck(lam, T):
    """Planck spectral radiance B_lambda in W / (sr m^3); lam in m, T in K."""
    h, c, kB = const.h, const.c, const.k
    return 2 * h * c**2 / lam**5 / np.expm1(h * c / (lam * kB * T))

lam = np.linspace(100e-9, 3000e-9, 1000)       # m
temps = [3000, 4000, 5000, 6000]               # K

fig, ax = plt.subplots()
for T in temps:
    ax.plot(lam * 1e9, planck(lam, T), label=f"{T} K")
ax.set_xlabel("wavelength (nm)")
ax.set_ylabel(r"$B_\lambda$ (W sr$^{-1}$ m$^{-3}$)")
ax.legend(title="temperature")
plt.show()

On linear axes the 6000 K curve dominates and the 3000 K curve is squashed near zero. When data span many orders of magnitude, use **logarithmic axes**: `ax.set_xscale("log")` and/or `ax.set_yscale("log")`. (There are also shortcuts `ax.semilogy`, `ax.semilogx`, and `ax.loglog` that plot and set the scales in one step.)

In [ ]:
lam = np.logspace(-7, -4, 500)       # 100 nm to 100 micrometres
fig, ax = plt.subplots()
for T in temps:
    ax.plot(lam * 1e9, planck(lam, T), label=f"{T} K")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_ylim(1e3, 1e14)
ax.set_xlabel("wavelength (nm)")
ax.set_ylabel(r"$B_\lambda$ (W sr$^{-1}$ m$^{-3}$)")
ax.legend()
plt.show()

Now every curve is visible, and you can see a nice feature: at long wavelength all curves become straight lines with the same slope. On a log-log plot, a **power law** $y = A x^p$ appears as a straight line with slope $p$, because $\log y = \log A + p \log x$. Here $B_\lambda \propto T/\lambda^4$ at long wavelength (the classical Rayleigh–Jeans law), so the slope is $-4$. Log-log plots will be your main tool for studying numerical errors starting next week.

Notice `np.logspace` for the grid: on a log axis you want points evenly spaced in $\log\lambda$, otherwise the left part of the plot gets very few points.

### Your turn 2

On one set of log-log axes, plot $B_\lambda$ at $T = 5800$ K together with the Rayleigh–Jeans approximation $B_\text{RJ} = 2ck_BT/\lambda^4$. Use a solid line for Planck and a dashed line for Rayleigh–Jeans. Where do they agree, and where does Rayleigh–Jeans fail badly (the "ultraviolet catastrophe")? Add a vertical line at the Wien peak.

In [ ]:
# Your code here

## 4. Vertical lines: the hydrogen spectrum

**Physics recap.** In the Bohr model, hydrogen's electron has energies $E_n = -13.6\ \mathrm{eV}/n^2$. A drop from level $n$ to a lower level $m$ emits a photon of energy $E_n - E_m$ and wavelength $\lambda = hc/(E_n - E_m)$. Transitions down to $m = 2$ (the Balmer series) fall in the visible.

A spectrum of discrete lines is naturally drawn as vertical lines: `ax.vlines(x, ymin, ymax)` draws one vertical line at each value in the array `x`.

In [ ]:
E1_eV = 13.6
hc_eVnm = const.h * const.c / const.e * 1e9      # h*c in eV nm (about 1240)

def hydrogen_wavelength_nm(n_upper, n_lower):
    """Wavelength in nm of the photon from the n_upper -> n_lower transition."""
    dE = E1_eV * (1 / n_lower**2 - 1 / n_upper**2)
    return hc_eVnm / dE

fig, ax = plt.subplots(figsize=(8, 2.5))
series = {"Lyman (n→1)": 1, "Balmer (n→2)": 2, "Paschen (n→3)": 3}
for i, (name, m) in enumerate(series.items()):
    n = np.arange(m + 1, m + 8)
    ax.vlines(hydrogen_wavelength_nm(n, m), 0, 1, color=f"C{i}", label=name)
ax.set_xscale("log")
ax.set_yticks([])                 # the height means nothing here, so hide the y axis
ax.set_xlabel("wavelength (nm)")
ax.legend(loc="upper left", fontsize=8)
plt.show()

New pieces: `figsize=(width, height)` in inches sets the figure size; `ax.set_yticks([])` removes tick marks; `legend(loc=..., fontsize=...)` positions the legend. Each series bunches up toward a *series limit* at short wavelength, as $n\to\infty$. (Recall `enumerate` gives you a counter `i` along with each item, and `.items()` loops over a dictionary's key–value pairs.)

The visible window is roughly 380–750 nm. You can shade it with `ax.axvspan`:

In [ ]:
n = np.arange(3, 10)
lam_balmer = hydrogen_wavelength_nm(n, 2)

fig, ax = plt.subplots(figsize=(8, 2.5))
ax.axvspan(380, 750, color="0.9", label="visible")    # "0.9" is a light gray
ax.vlines(lam_balmer, 0, 1, color="k")
for lam_i, n_i in zip(lam_balmer[:4], n[:4]):
    ax.text(lam_i, 1.05, f"{n_i}→2", ha="center", fontsize=8)
ax.set_ylim(0, 1.2)
ax.set_xlim(350, 700)
ax.set_yticks([])
ax.set_xlabel("wavelength (nm)")
ax.set_title("Balmer series of hydrogen")
plt.show()

`ax.text(x, y, "string")` writes text at a data position; `ha="center"` centers it horizontally.

## 5. Scatter plots, error bars, and real data

From here on we use the course data files. Run the helper cell once:

In [ ]:
import os
DATA_URL = "https://raw.githubusercontent.com/burzynski-lab/OU_PHYS2222_ComputationalPhysics/main/Data/"

def data_path(filename):
    """Return a local path to a course data file if it exists, otherwise its GitHub URL."""
    for folder in ["../Data", "Data", "."]:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            return path
    return DATA_URL + filename

### 5.1 Millikan's photoelectric data

**Physics recap.** Light shining on a metal can eject electrons. Einstein's explanation: each photon of frequency $f$ carries energy $hf$; an electron that absorbs it escapes with kinetic energy at most $hf - \phi$, where the work function $\phi$ is the energy needed to leave the metal. That maximum kinetic energy is measured by the "stopping voltage" needed to halt the electrons: $eV_\text{stop} = hf - \phi$. So $V_\text{stop}$ vs $f$ should be a **straight line with slope $h/e$**. Millikan's 1916 measurements of this line were among the first precise determinations of $h$.

The file has two columns, frequency (Hz) and stopping voltage (V).

In [ ]:
freq, V_stop = np.loadtxt(data_path("millikan.txt"), unpack=True)

fig, ax = plt.subplots()
ax.scatter(freq, V_stop)
ax.set_xlabel("frequency (Hz)")
ax.set_ylabel("stopping voltage (V)")
plt.show()

`ax.scatter(x, y)` draws a marker at each point without connecting lines (`ax.plot(x, y, "o")` does nearly the same). Use markers for measured data and lines for theory curves: a line between data points suggests you know what happens between them.

The file has no uncertainties. To show how **error bars** work, let's *assume* each voltage is uncertain by 0.05 V (an illustrative value, not Millikan's). `ax.errorbar(x, y, yerr=...)` draws them; `fmt="o"` means markers and no connecting line, and `capsize` adds little caps.

We also preview **fitting**: `np.polyfit(x, y, 1)` finds the straight line $y = ax + b$ that best matches the points (in the "least squares" sense, which you will learn properly in Week 11). It returns the coefficients, highest power first.

In [ ]:
sigma_V = 0.05 * np.ones_like(V_stop)        # assumed uncertainty (V), same shape as V_stop
slope, intercept = np.polyfit(freq, V_stop, 1)

f_line = np.linspace(0, 1.3e15, 100)
fig, ax = plt.subplots()
ax.errorbar(freq, V_stop, yerr=sigma_V, fmt="o", capsize=3, label="Millikan (1916)")
ax.plot(f_line, slope * f_line + intercept, "--", label="straight-line fit")
ax.axhline(0, color="gray", lw=0.8)
ax.set_xlabel("frequency $f$ (Hz)")
ax.set_ylabel(r"stopping voltage $V_\mathrm{stop}$ (V)")
ax.legend()
plt.show()

print(f"slope h/e = {slope:.4e} V s   (accepted: {const.h / const.e:.4e} V s)")
print(f"h = {slope * const.e:.4e} J s   (accepted: {const.h:.4e} J s)")
print(f"work function = {-intercept:.2f} eV")

From six data points over a century old you get Planck's constant to within about 1%. The intercept is $-\phi/e$, so it gives the work function of the metal surface: a couple of eV, typical of alkali metals like the sodium Millikan used. The line crosses $V=0$ at the threshold frequency, below which no electrons come out no matter how bright the light.

### Your turn 3

Make a two-part figure from the Millikan data: in the top panel the data and the fit (as above); in the bottom panel the **residuals** $V_\text{stop} - (\text{slope}\cdot f + \text{intercept})$ with the same assumed error bars and a horizontal line at zero. Use `fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True)` (explained in Section 7). Do the residuals look random, or is there a pattern?

In [ ]:
# Your code here

## 6. Histograms

A **histogram** divides the range of a quantity into bins and counts how many values fall in each bin. It is the standard way to see the *distribution* of many measurements.

As an example, simulate 2000 measurements of $g$ with a Gaussian ("normal") spread of 0.05 m/s² around 9.81 m/s². (Random numbers are the topic of Week 9; for now `rng.normal(mean, std, size)` just gives us realistic-looking scatter.)

In [ ]:
rng = np.random.default_rng(2027)
g_sim = rng.normal(9.81, 0.05, size=2000)      # simulated measurements, m/s^2

fig, ax = plt.subplots()
counts, edges, patches = ax.hist(g_sim, bins=30)
ax.set_xlabel(r"measured $g$ (m/s$^2$)")
ax.set_ylabel("measurements per bin")
plt.show()
print("number of bins:", len(counts), "  bin width:", edges[1] - edges[0])

`ax.hist` returns the counts in each bin and the bin edges (one more edge than bins), which is useful for further calculations.

Key options:

- `bins=30` sets the number of bins, or pass an array of edges, e.g. `bins=np.linspace(9.6, 10.0, 41)`. Too few bins hide structure; too many make every bin noisy.
- `density=True` rescales so the total area is 1. Then the histogram estimates a **probability density** and can be compared with a formula. For a Gaussian the density is
$$p(x) = \frac{1}{\sigma\sqrt{2\pi}}\,e^{-(x-\mu)^2/2\sigma^2}.$$
- `histtype="step"` draws only the outline, good for overlaying several histograms.

In [ ]:
mu, sigma = 9.81, 0.05
xg = np.linspace(9.6, 10.0, 300)
pdf = np.exp(-(xg - mu)**2 / (2 * sigma**2)) / (sigma * np.sqrt(2 * np.pi))

fig, ax = plt.subplots()
ax.hist(g_sim, bins=np.linspace(9.6, 10.0, 41), density=True, alpha=0.6, label="simulated data")
ax.plot(xg, pdf, "k-", label="Gaussian, $\\mu=9.81$, $\\sigma=0.05$")
ax.set_xlabel(r"measured $g$ (m/s$^2$)")
ax.set_ylabel(r"probability density (s$^2$/m)")
ax.legend()
plt.show()

Note the units of a probability density: (1/units of $x$), so that density times bin width is a probability.

### Your turn 4

Radioactive nuclei decay at random times. For a sample with mean lifetime $\tau$, the decay times follow $p(t) = e^{-t/\tau}/\tau$. Simulate 5000 decay times of muons ($\tau = 2.2\ \mu$s) with `rng.exponential(2.2, size=5000)` (in μs). Plot a normalized histogram (`density=True`) with `histtype="step"` and overlay $p(t)$. Then make the $y$ axis logarithmic. What shape does the exponential become, and why?

In [ ]:
# Your code here

## 7. Several panels: subplots

`plt.subplots(nrows, ncols)` makes a grid of axes. With more than one panel, `ax` becomes an *array* of axes, which you index like any NumPy array. Useful options:

- `figsize=(w, h)` makes room for the panels;
- `sharex=True` / `sharey=True` make panels share the same axis range (and hide repeated tick labels);
- `fig.tight_layout()` adjusts spacing so labels do not overlap (or use `layout="constrained"` when creating the figure).

### 7.1 Sunspots: a long time series

The number of dark spots on the Sun has been recorded monthly since 1749. The file has two columns: months since January 1749, and the monthly sunspot number. Sunspots trace the Sun's magnetic activity, which rises and falls in a cycle of roughly a decade.

In [ ]:
month, spots = np.loadtxt(data_path("sunspots.txt"), unpack=True)
year = 1749 + month / 12
print(f"{len(year)} months from {year[0]:.0f} to {year[-1]:.1f}")

fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharey=True)
axes[0].plot(year, spots, lw=0.6)
axes[0].set_title("Full record")

recent = (year >= 1950) & (year < 2000)        # boolean mask, from notebook 04
axes[1].plot(year[recent], spots[recent], lw=0.8, color="C1")
axes[1].set_title("1950–2000")

for ax in axes:
    ax.set_xlabel("year")
    ax.set_ylabel("sunspot number")
fig.tight_layout()
plt.show()

The zoomed panel makes the cycle obvious: count the peaks between 1950 and 2000 and you will estimate an ~11-year period. (Measuring this period precisely with a Fourier transform is a possible final project.)

Monthly values are noisy. A common trick is a running average. Here is one way with `np.convolve`, which averages each point with its neighbours; you do not need to understand its details yet:

In [ ]:
window = 13                                        # months
smooth = np.convolve(spots, np.ones(window) / window, mode="same")

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(year, spots, color="0.75", lw=0.5, label="monthly")
ax.plot(year, smooth, color="C0", lw=1.2, label=f"{window}-month running mean")
ax.set_xlim(1900, year[-1])
ax.set_xlabel("year")
ax.set_ylabel("sunspot number")
ax.legend(loc="upper left")
plt.show()

Gray for raw data and a strong color for the smoothed curve is a clean way to show both without clutter.

### 7.2 Global temperatures: missing data and `np.genfromtxt`

NASA's GISTEMP record gives the global mean surface temperature **anomaly** (difference from the 1951–1980 average, in °C) for each month since 1880. Let's look at the top of the file first.

In [ ]:
import urllib.request, os

path = data_path("GLB.Ts+dSST.csv")
if path.startswith("http"):
    lines = urllib.request.urlopen(path).read().decode().splitlines()
else:
    with open(path) as f:
        lines = f.read().splitlines()
for line in lines[:4]:
    print(line)
print("...")
print(lines[-1])

(Do not worry about the details of that cell; it just prints the first and last few lines whether the file is local or online.)

Observations: line 0 is a title, line 1 holds the column names, values are comma-separated, and missing values are written `***` (e.g. December of the latest year, not yet measured). `np.loadtxt` would crash on `***`. `np.genfromtxt` is more forgiving: anything it cannot convert to a number becomes `nan`.

Columns: 0 is Year, 1–12 are January–December, 13 is `J-D`, the January–December annual mean.

In [ ]:
temp = np.genfromtxt(data_path("GLB.Ts+dSST.csv"), delimiter=",", skip_header=2)
print(temp.shape)
years = temp[:, 0]
annual = temp[:, 13]
monthly = temp[:, 1:13]        # 2D: one row per year, 12 columns
print("last rows of annual mean:", annual[-3:])
print("number of missing monthly values:", np.sum(np.isnan(monthly)))

`np.isnan` gives a mask of the missing values. Many NumPy functions return `nan` if any input is `nan`; the `nan`-aware versions `np.nanmean`, `np.nanmax`, etc. skip them.

In [ ]:
monthly_mean = np.nanmean(monthly, axis=1)      # mean over the months that exist
ok = ~np.isnan(annual)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(years, monthly_mean, "o", ms=3, color="C0", alpha=0.5, label="annual mean")
smooth_T = np.convolve(annual[ok], np.ones(5) / 5, mode="valid")
ax.plot(years[ok][2:-2], smooth_T, color="C3", lw=2, label="5-year running mean")
ax.axhline(0, color="gray", lw=0.8)
ax.set_xlabel("year")
ax.set_ylabel("temperature anomaly (°C)\nrelative to 1951–1980")
ax.set_title("Global surface temperature (NASA GISTEMP)")
ax.legend()
plt.show()

(`"\n"` inside a string starts a new line, handy for long axis labels.)

**A first look at pandas.** For tables with named columns and missing values, the `pandas` library is often more convenient than raw NumPy. You will use it more in Unit 3; here is just a taste. `pd.read_csv` reads the file, `na_values="***"` marks the missing entries, and you select columns by name.

In [ ]:
import pandas as pd

df = pd.read_csv(data_path("GLB.Ts+dSST.csv"), skiprows=1, na_values="***")
print(df[["Year", "Jan", "Jul", "J-D"]].tail())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(df["Year"], df["Jan"], lw=1, label="January")
ax.plot(df["Year"], df["Jul"], lw=1, ls="--", label="July")
ax.set_xlabel("year")
ax.set_ylabel("anomaly (°C)")
ax.legend()
plt.show()

## 8. Two-dimensional data: images and contours

Many physics quantities depend on two variables: a potential $V(x, y)$, a wavefunction $\psi(x, y)$, the intensity on a detector. To show them we need a **grid** of points.

### 8.1 Making a grid with `np.meshgrid`

`np.meshgrid(x, y)` takes a 1D array of $x$ values and one of $y$ values and returns two 2D arrays, `X` and `Y`, holding the $x$ and $y$ coordinate of every grid point. Then any formula in `X` and `Y` gives a 2D array of function values, thanks to vectorization.

In [ ]:
x_small = np.array([0.0, 1.0, 2.0])
y_small = np.array([10.0, 20.0])
X, Y = np.meshgrid(x_small, y_small)
print("X =\n", X)
print("Y =\n", Y)
print("X + Y =\n", X + Y)

Each row of `X` is a copy of `x`, and each column of `Y` is a copy of `y`. So `X[i, j], Y[i, j]` is the point in row $i$ (the $i$-th $y$ value) and column $j$ (the $j$-th $x$ value).

### 8.2 A quantum particle in a 2D box

**Physics recap.** A particle trapped in a square box of side $L$ with impenetrable walls (the 2D version of the infinite square well) has stationary states labelled by two quantum numbers $n_x, n_y = 1, 2, 3, \dots$:

$$\psi_{n_x n_y}(x, y) = \frac{2}{L}\,\sin\!\left(\frac{n_x\pi x}{L}\right)\sin\!\left(\frac{n_y \pi y}{L}\right), \qquad E \propto n_x^2 + n_y^2.$$

The probability density of finding the particle at $(x, y)$ is $|\psi|^2$. Let's show it as an image.

In [ ]:
L = 1.0                                   # box size (nm)
x = np.linspace(0, L, 200)
y = np.linspace(0, L, 200)
X, Y = np.meshgrid(x, y)

def psi_box(X, Y, nx, ny, L=1.0):
    return (2 / L) * np.sin(nx * np.pi * X / L) * np.sin(ny * np.pi * Y / L)

prob = psi_box(X, Y, 2, 1) ** 2

fig, ax = plt.subplots()
im = ax.imshow(prob, origin="lower", extent=[0, L, 0, L], cmap="viridis")
fig.colorbar(im, ax=ax, label=r"$|\psi|^2$ (nm$^{-2}$)")
ax.set_xlabel("$x$ (nm)")
ax.set_ylabel("$y$ (nm)")
ax.set_title(r"$n_x=2,\ n_y=1$")
plt.show()

Things to know about `imshow`:

- It treats the array as a picture: by default row 0 is drawn at the **top**. `origin="lower"` puts row 0 at the bottom, as physicists expect for a $y$ axis.
- `extent=[xmin, xmax, ymin, ymax]` labels the axes with physical coordinates instead of pixel numbers.
- `fig.colorbar(im, ax=ax, label=...)` adds the color scale. **An image without a colorbar is like a plot without a $y$ axis.**

`ax.pcolormesh(X, Y, Z)` does a similar job and works even when the grid is not evenly spaced. `ax.contour(X, Y, Z)` draws lines of constant value, like a topographic map, and `ax.contourf` fills between them. Below, four states side by side, using `pcolormesh`, with the same color scale on all panels (`vmin`, `vmax`) so they can be compared.

In [ ]:
states = [(1, 1), (2, 1), (1, 2), (2, 2)]
fig, axes = plt.subplots(1, 4, figsize=(12, 3.2), sharey=True, layout="constrained")
for ax, (nx, ny) in zip(axes, states):
    pc = ax.pcolormesh(X, Y, psi_box(X, Y, nx, ny) ** 2, vmin=0, vmax=16, cmap="viridis")
    ax.set_title(f"$n_x={nx},\\ n_y={ny}$")
    ax.set_xlabel("$x$ (nm)")
    ax.set_aspect("equal")
axes[0].set_ylabel("$y$ (nm)")
fig.colorbar(pc, ax=axes, label=r"$|\psi|^2$ (nm$^{-2}$)")
plt.show()

States $(2,1)$ and $(1,2)$ have the same energy ($n_x^2+n_y^2 = 5$) but different shapes: this is a **degeneracy**, caused by the symmetry of the square box. The dark lines where $|\psi|^2 = 0$ are **nodal lines**; higher-energy states have more of them.

`ax.set_aspect("equal")` makes one unit in $x$ the same length as one unit in $y$, so a square box looks square.

### 8.3 Contours of a potential

Contour plots are natural for potentials. The electric potential of two charges $+q$ and $-q$ (an electric dipole) at positions $(\pm d/2, 0)$ is
$$V(x, y) = kq\left(\frac{1}{r_+} - \frac{1}{r_-}\right),$$
where $r_\pm$ is the distance to each charge. Lines of constant $V$ are equipotentials, and the electric field points perpendicular to them.

In [ ]:
xg = np.linspace(-2, 2, 300)
yg = np.linspace(-1.5, 1.5, 300)
X2, Y2 = np.meshgrid(xg, yg)
d = 1.0
r_plus = np.sqrt((X2 - d / 2)**2 + Y2**2)
r_minus = np.sqrt((X2 + d / 2)**2 + Y2**2)
V = 1 / r_plus - 1 / r_minus                     # in units of kq / (1 m)

levels = np.linspace(-4, 4, 17)
fig, ax = plt.subplots(figsize=(7, 5))
cf = ax.contourf(X2, Y2, V, levels=levels, cmap="RdBu_r", extend="both")
ax.contour(X2, Y2, V, levels=levels, colors="k", linewidths=0.5)
fig.colorbar(cf, ax=ax, label=r"$V$ (units of $kq$/m)")
ax.plot([d / 2, -d / 2], [0, 0], "ko")
ax.set_aspect("equal")
ax.set_xlabel("$x$ (m)")
ax.set_ylabel("$y$ (m)")
plt.show()

Here a **diverging** colormap (`"RdBu_r"`: red for positive, blue for negative, white near zero) is appropriate because the sign matters. For quantities that are always positive, use a sequential map like `"viridis"`. `extend="both"` adds arrows to the colorbar showing that values beyond $\pm 4$ (near the charges) are clipped.

### Your turn 5

Plot $|\psi|^2$ for the 2D box state $(n_x, n_y) = (3, 2)$ using `contourf` with a colorbar, and overlay the contour line at a small value (e.g. `levels=[0.05]`) to highlight the nodal lines. Then compute the energies $n_x^2 + n_y^2$ for all $n_x, n_y$ from 1 to 4 (broadcasting from notebook 04 helps) and find which energy levels are degenerate.

In [ ]:
# Your code here

## 9. Saving figures

`fig.savefig(filename)` writes the figure to a file. The format comes from the extension:

- `.png` for slides and web pages; set the resolution with `dpi=` (dots per inch, 150–300 is good);
- `.pdf` or `.svg` for reports: these are *vector* formats that stay sharp at any zoom.

`bbox_inches="tight"` trims extra white space. Call `savefig` **before** `plt.show()` (in a script, `show` can leave you with an empty figure).

In [ ]:
import tempfile

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(beta, K_rel, label="relativistic")
ax.plot(beta, K_cl, "--", label="classical")
ax.set_xlabel(r"$\beta$")
ax.set_ylabel("kinetic energy (MeV)")
ax.legend()

out_png = os.path.join(tempfile.gettempdir(), "kinetic_energy.png")
out_pdf = os.path.join(tempfile.gettempdir(), "kinetic_energy.pdf")
fig.savefig(out_png, dpi=200, bbox_inches="tight")
fig.savefig(out_pdf, bbox_inches="tight")
plt.show()
print("saved", out_png, os.path.getsize(out_png), "bytes")

In Colab, to keep a figure, save it and then download it from the Files panel (folder icon on the left); a temporary folder like the one above is just for demonstration.

## 10. What makes a good scientific figure?

A figure in a lab report or paper should be understandable on its own. A checklist (this is also how your project figures will be graded):

1. **Axis labels with units** on every axis, e.g. "wavelength (nm)".
2. **Readable text.** Tick labels and axis labels should be about as large as the text around the figure. Increase with `fontsize=` or globally with `plt.rcParams["font.size"] = 12`.
3. **A legend** whenever there is more than one curve, or label curves directly with `ax.text`.
4. **Data as markers (with error bars), theory as lines.**
5. **Colorblind-friendly colors**, and differences in line style as well as color. Use `viridis`-like colormaps for images, and always add a colorbar.
6. **Not too much in one panel.** More than about 5 curves becomes spaghetti; split into subplots instead.
7. **The right scale**: log axes when values span orders of magnitude; zoom on the interesting region.
8. **No chart junk**: no 3D effects, no unnecessary colors or boxes.

Here is the same information drawn badly and then well:

In [ ]:
temps_many = np.arange(3000, 11000, 1000)
lam = np.linspace(100e-9, 3000e-9, 500)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ax = axes[0]
for T in temps_many:
    ax.plot(lam, planck(lam, T))
ax.set_title("Bad: no units, no legend, 8 curves")

ax = axes[1]
cmap = plt.get_cmap("plasma")
for k, T in enumerate([3000, 5000, 7000, 9000]):
    ax.plot(lam * 1e9, planck(lam, T), color=cmap(k / 4), label=f"{T} K")
ax.set_yscale("log")
ax.set_ylim(1e10, 1e15)
ax.set_xlabel("wavelength (nm)", fontsize=12)
ax.set_ylabel(r"$B_\lambda$ (W sr$^{-1}$ m$^{-3}$)", fontsize=12)
ax.legend(title="$T$")
ax.set_title("Better")
fig.tight_layout()
plt.show()

In the better version the colors also come from a sequential colormap (`plt.get_cmap("plasma")`, then `cmap(value)` with a value between 0 and 1), so the color order matches the temperature order. That is a nice touch when the curves form a sequence.

### Your turn 6

Pick one figure you made in this notebook (or in Your turn 1–5) and improve it using the checklist above. Then save it as both a PNG (dpi 200) and a PDF. Write two sentences in a Markdown cell explaining what you changed and why.

In [ ]:
# Your code here

## Summary

- `import matplotlib.pyplot as plt`; make figures with `fig, ax = plt.subplots()` (OO style) and call methods on `ax`.
- Lines and points: `ax.plot(x, y, color=, ls=, lw=, marker=, label=)`, `ax.scatter`, `ax.errorbar(x, y, yerr=, fmt="o", capsize=)`.
- Reference lines and regions: `ax.axhline`, `ax.axvline`, `ax.vlines`, `ax.axvspan`; text with `ax.text`.
- Labels: `ax.set_xlabel`, `ax.set_ylabel`, `ax.set_title`, `ax.legend()`; LaTeX in raw strings `r"$\lambda$ (nm)"`. Always include units.
- Ranges and scales: `ax.set_xlim`, `ax.set_ylim`, `ax.set_xscale("log")`, `ax.set_yscale("log")`; on log-log axes a power law is a straight line.
- Histograms: `ax.hist(data, bins=, density=, histtype="step")`.
- Panels: `fig, axes = plt.subplots(nrows, ncols, figsize=, sharex=, sharey=)`, `fig.tight_layout()`.
- 2D data: `X, Y = np.meshgrid(x, y)`; `ax.imshow(Z, origin="lower", extent=)`, `ax.pcolormesh(X, Y, Z)`, `ax.contour`, `ax.contourf`; always `fig.colorbar(...)`.
- Data: `np.loadtxt`, `np.genfromtxt` (missing values become `nan`; use `np.isnan`, `np.nanmean`), `pd.read_csv(..., na_values=)`.
- Quick straight-line fit: `slope, intercept = np.polyfit(x, y, 1)` (more in Week 11).
- Save: `fig.savefig("name.png", dpi=200, bbox_inches="tight")` or `.pdf`.

## Further reading

- Matplotlib [Quick start guide](https://matplotlib.org/stable/users/explain/quick_start.html) and the [gallery](https://matplotlib.org/stable/gallery/index.html) of examples (find a plot like the one you want and adapt its code).
- Matplotlib documentation on [choosing colormaps](https://matplotlib.org/stable/users/explain/colors/colormaps.html).
- J. VanderPlas, *Python Data Science Handbook*, Chapter 4 "Visualization with Matplotlib" ([free online](https://jakevdp.github.io/PythonDataScienceHandbook/)).
- M. Newman, *Computational Physics*, Chapter 3 (graphics and visualization).